# Custom prediction with a trained model

This notebooks shows how to use a fine-tuned model for prediction tasks.

## Imports

In [ ]:
import os
import json
# Set the GPU to the one with available memory (nvidia-smi)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
from transformers import AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from esnlir.dataset_utils.dataset import BERTDataset
from torch.utils.data import DataLoader
from tqdm import tqdm

import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, classification_report

import pandas as pd

## Constants

In [ ]:
# File path of the dataset you want to predict
DATASET = "/data/TESIS/nli-refactor/data/base_dataset/test.json"

# Folder path of the model you fine-tuned
MODEL = "/data/TESIS/nli-refactor/model/roberta/model"

In [ ]:
# Maximum length of tokens per sentence
MAX_LEN = 256

# Original model from hugging-faces
MODEL_TYPE = "bertin-project/bertin-roberta-base-spanish" #"FacebookAI/xlm-roberta-base"

# Batch size
BATCH_SIZE = 32

# CUDA device
DEVICE = "cuda:0"

# CPU device
CPU_DEVICE = "cpu"

In [ ]:
# True labels output file
Y_TRUE_FILE = "./y_true.json"

# Original dataset output file
DF_TEST_FILE = "./df_test.json"

# Predicted labels output file
Y_PRED_FILE = "./y_pred_bertin.json"

## Execution

### Load the dataset

In [ ]:
dataset = BERTDataset(
    dataframe_file=DATASET,
    max_len=MAX_LEN,
    model_type=MODEL_TYPE,
    only_premise=False,
    max_samples=None
)

In [ ]:
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE)

## Load the model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL).to(DEVICE)

## Predict the dataset with the model

In [ ]:
#
model.zero_grad()

# Real labels accumulator
y_true = []
y_pred = []
# Run over all dataset batches
for batch in tqdm(dataloader):
    
    # The model tokenized input
    inputs = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    
    # Get the labels to evaluate
    batch_y_true = batch["labels"].to(CPU_DEVICE).detach().numpy().tolist()
    y_true.extend(batch_y_true)
    
    # Predict using the model
    batch_y_pred = model(inputs, attention_mask=attention_mask).logits.to(CPU_DEVICE).detach().numpy().tolist()
    y_pred.extend(batch_y_pred)
    
    torch.cuda.empty_cache()

In [ ]:
y_true[:2]

In [ ]:
y_pred[:2]

# Save predictions and original dataset

In [ ]:
def save_json(data, path):
    with open(path, "w") as f:
        json.dump(data, f)

In [ ]:
save_json(y_true, Y_TRUE_FILE)

In [ ]:
save_json(y_pred, Y_PRED_FILE)

In [ ]:
pd.read_json(DATASET, lines=True).sort_values("connector_type").to_json(DF_TEST_FILE, lines=True, orient="records")